In [1]:
from cluster_paths import get_cluster_paths
scratch_path, repo_path, _ = get_cluster_paths()

%load_ext autoreload
%autoreload 2

import h5py, os, yaml
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt

import tensorflow as tf
for gpu in tf.config.list_physical_devices(device_type="GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

from tqdm import tqdm
from trianglechain import TriangleChain

from msfm.utils import files, observation, catalog, buzzard

from deep_lss.models.grid_model import GridLossModel
from deep_lss.utils import configuration, evaluation
from deep_lss.nets.mlp import MultiLayerPerceptron

from msi.utils import preprocessing, dataset

2026-05-15 14:18:30.402749: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-15 14:18:30.411750: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-15 14:18:30.415910: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/users/athomsen/dlss/tf_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# v16

In [ ]:
# msfm_conf = files.load_config(f"{repo_path}/multiprobe-simulation-forward-model/configs/v16/default.yaml")
msfm_conf = files.load_config(f"{repo_path}/multiprobe-simulation-forward-model/configs/v16/default.yaml")

# data_dir = f"{scratch_path}/v11desy3/v16/default"
# data_dir = f"{scratch_path}/deep_lss/data/v16/default"
data_dir = f"{scratch_path}/deep_lss/data/v16/rot_in_place"

# out_dir = f"{scratch_path}/deep_lss/v16/default/cls"
# out_dir = f"{scratch_path}/deep_lss/runs/v16/default/cls"
# out_dir = f"{scratch_path}/deep_lss/runs/v16/no_sc/cls"
out_dir = f"{scratch_path}/deep_lss/runs/v16/rot_in_place/cls"
# model_name = "v1"
# model_name = "v3/24Mpc"
# model_name = "v3/28Mpc"
# model_name = "v3/32Mpc"
# model_name = "v3/36Mpc"
# model_name = "v5_small"
model_name = "v5_diag"

In [ ]:
# # lensing
# dlss_conf = configuration.load_deep_lss_config(
#     f"{repo_path}/y3-deep-lss/configs/v16/default/lensing/dlss.yaml"
# )

In [ ]:
# # clustering
# dlss_conf = configuration.load_deep_lss_config(
#    f"{repo_path}/y3-deep-lss/configs/v16/default/clustering/dlss.yaml"
# )

In [ ]:
# # cross
# dlss_conf = configuration.load_deep_lss_config(
#    ff"{repo_path}/y3-deep-lss/configs/v16/default/cross/dlss.yaml"
# )

In [ ]:
# combined
dlss_conf = configuration.load_deep_lss_config(
   ff"{repo_path}/y3-deep-lss/configs/v16/default/combined/dlss.yaml"
)

# training

In [ ]:
params = dlss_conf["dset"]["training"]["params"]
with_lensing = dlss_conf["dset"]["common"]["with_lensing"]
with_clustering = dlss_conf["dset"]["common"]["with_clustering"]
with_cross_z = True

apply_log = True
standardize = False
batch_size = 2**10
# n_steps = 100_000
n_steps = 500_000

rng = np.random.default_rng(12)

# strategy = tf.distribute.MirroredStrategy()
# print("replicas =", strategy.num_replicas_in_sync)

if with_lensing and not with_clustering:
    probe = "lensing"
    with_cross_probe = False
elif with_clustering and not with_lensing:
    probe = "clustering"
    with_cross_probe = False
elif with_lensing and with_clustering:
    probe = "combined"
    with_cross_probe = True
elif not with_lensing and not with_clustering:
    probe = "cross"
    with_cross_probe = True

pred_dir = os.path.join(out_dir, probe, model_name)
os.makedirs(pred_dir, exist_ok=True)    

pred_file = os.path.join(pred_dir, f"preds_{n_steps}.h5")

print("out_dir = ", out_dir)
print("pred_file = ", pred_file)

net_conf = {}
with open(os.path.join(pred_dir, "configs.yaml"), "w") as f:
    yaml.dump_all([net_conf, dlss_conf, msfm_conf], f)

## data

In [ ]:
cl_dset_train, cl_dset_test, out_dict = dataset.get_binned_power_spectra_dset(
    data_dir, 
    # configuration
    msfm_conf=msfm_conf,
    dlss_conf=dlss_conf,
    params=params,
    # selection
    with_lensing=with_lensing,
    with_clustering=with_clustering,
    with_cross_z=with_cross_z,
    with_cross_probe=with_cross_probe,
    # dset
    batch_size=batch_size,
    # preprocessing
    apply_log=apply_log,
    standardize=standardize,
)

## network

In [ ]:
n_cls = out_dict["grid/cls/train"].shape[-1]
n_params = len(params)
n_summary = 2 * n_params

num_hidden_units = 1024
num_layers = 2

dropout_rate = 0.1

def get_cl_summary_network(n_summary=n_summary):
    mlp = MultiLayerPerceptron(
        output_size=n_summary, 
        num_hidden_units=num_hidden_units, 
        num_layers=num_layers, 
        dropout_rate=dropout_rate
    )
    mlp.build((None, n_cls))
    mlp.summary()
    
    return mlp

In [ ]:
def train_model(model, eval_every=None):
    do_validation = eval_every is not None
    
    train_losses = []
    train_steps = []
    vali_steps = []
    vali_losses = []
    for i, (cl_batch, cosmo_batch) in tqdm(enumerate(cl_dset_train), total=n_steps + 1):
        if i > n_steps:
            break

        loss = model.grid_train_step(cl_batch, cosmo_batch)
        train_losses.append(float(loss.numpy()))
        train_steps.append(i)
        
        # if do_validation and (i % eval_every == 0) and (i !=0):
        #     vali_loss = []
        #     for cl_batch, cosmo_batch in cl_dset_test:
        #         vali_loss.append(model.vali_loss_fn(model(cl_batch, training=False), cosmo_batch))
        #     vali_losses.append(np.mean(vali_loss))
        #     vali_steps.append(i)

    fig, ax = plt.subplots()
    ax.plot(train_steps[100:], train_losses[100:], label="training")
    ax.plot(vali_steps, vali_losses, label="validation")
    ax.legend()
    
    if do_validation and len(vali_losses) > 0:
        print(f"final validation loss = {vali_losses[-1]}")
        
    model.save_model()

## VMIM

In [ ]:
learning_rate = 1e-4

# with strategy.scope():
optimizer = tf.keras.optimizers.Adam(learning_rate)

summary_net = get_cl_summary_network()

model = GridLossModel(
    summary_net,
    n_side=None,
    indices=None,
    optimizer=optimizer,
    checkpoint_dir=os.path.join(pred_dir, "network/checkpoint"),
    summary_dir=os.path.join(pred_dir, "network/history"),
    # strategy=strategy,
    restore_checkpoint=True,
    # z_bank_size=4*batch_size,
)

# model.restore_model_from_checkpoint_path(
#     os.path.join(pred_dir, "model", f"ckpt-{checkpoint_number}")
# )

model.setup_grid_loss_step(
    batch_size=batch_size,
    dim_theta=n_params,
    loss="mutual_info",
    dim_x=n_cls,
    dim_summary=n_summary,
    mutual_info_estimator="variational",
    clip_by_global_norm=1.0,
    mutual_info_kwargs={"full_covariance": False},
    # feature regularization
    # z_type="sw",
    # z_weight=10,
    # z_layer="last",
)

In [ ]:
# train_model(model, eval_every=n_steps//10)

# evaluation

## CosmoGrid

### grid

In [ ]:
grid_preds = model(out_dict["grid/cls/test"], training=False)
grid_cosmos = out_dict["grid/cosmos/test"]
# fidu_preds = model(out_dict["fidu/cls"], training=False)

with h5py.File(pred_file, "w") as f:
    f.create_dataset(name="grid/preds/test", data=grid_preds)
    f.create_dataset(name="grid/cosmos/test", data=grid_cosmos)
    # f.create_dataset(name="fidu/preds", data=fidu_preds)
print(f"Wrote to {pred_file}\n")


def plot_cls_space_prior_predictive(obs_cl, n_rand=100):
    i_rand = rng.integers(0, grid_preds.shape[0], n_rand)
    
    fig, ax = plt.subplots()
    ax.plot(out_dict["grid/cls/test"][i_rand,:].T, alpha=0.5)
    ax.plot(np.squeeze(obs_cl), alpha=1, color="k", linestyle=":")
    ax.legend(loc="best")
    ax.set(xlabel="data vec dim", ylabel=r"$C_\ell$")


In [ ]:
n_examples = 4

for i_grid in range(n_examples):
    # unique cosmological parameters
    i_grid *= msfm_conf["analysis"]["grid"]["n_perms_per_cosmo"] * msfm_conf["analysis"]["n_patches"]

    obs_label = f"grid_{i_grid}"
    obs_pred = grid_preds[i_grid]
    obs_cosmo = grid_cosmos[i_grid]
    
    # save
    evaluation.append_obs_to_file(pred_file, f"obs/preds/{obs_label}", obs_pred)
    evaluation.append_obs_to_file(pred_file, f"obs/cosmos/{obs_label}", obs_cosmo)


### benchmarks

In [ ]:
obs_labels = []
obs_files = []

obs_dir = os.path.join(data_dir, "obs")

# benchmark
obs_labels.append("bench_fidu")
obs_files.append(os.path.join(obs_dir, "fiducial_bench_obs_maps.h5"))

# obs_labels.append("bench_box")
# obs_files.append(os.path.join(obs_dir, "box_size_obs_maps.h5"))

# obs_labels.append("bench_particle")
# obs_files.append(os.path.join(obs_dir, "particle_count_obs_maps.h5"))

# obs_labels.append("bench_redshift")
# obs_files.append(os.path.join(obs_dir, "redshift_resolution_obs_maps.h5"))

# obs_labels.append("bench_bsc=rot")
# obs_files.append(os.path.join(obs_dir, "fiducial_bench_obs_maps_rot.h5"))

# obs_labels.append("bench_bsc=fit")
# obs_files.append(os.path.join(obs_dir, "fiducial_bench_obs_maps_bsc=fit.h5"))

# obs_labels.append("bench_bsc=0")
# obs_files.append(os.path.join(obs_dir, "fiducial_bench_obs_maps_bsc=0.h5"))

# obs_labels.append("bench_bsc=1")
# obs_files.append(os.path.join(obs_dir, "fiducial_bench_obs_maps_bsc=1.h5"))


In [ ]:
plot_diagnostics = True

for obs_label, obs_file in zip(obs_labels, obs_files):
    print(f"CosmoGrid mock: {obs_label}")
    
    with h5py.File(obs_file, "r") as f_in:
        obs_cls_raw = f_in["obs/cls_raw"][:]
        print("obs_cls_raw.shape =", obs_cls_raw.shape)
    
    # forward model
    obs_cl = preprocessing.get_preprocessed_cl_observation(
        obs_cl=obs_cls_raw,
        # configuration
        msfm_conf=msfm_conf,
        dlss_conf=dlss_conf,
        base_dir=data_dir,
        nest_in=False,
        # selection
        with_lensing=with_lensing,
        with_clustering=with_clustering,
        with_cross_z=with_cross_z,
        with_cross_probe=with_cross_probe,
        # additional preprocessing
        apply_log=apply_log,
        standardize=standardize,
        make_plot=False,
    )
    obs_cl = np.squeeze(obs_cl)

    # evaluate
    obs_pred = model(obs_cl, training=False).numpy()

    if plot_diagnostics:
        plot_cls_space_prior_predictive(obs_cl.T)
        evaluation.plot_summary_space_prior_predictive(grid_preds, obs_pred)

    # save
    obs_label = f"obs/preds/{obs_label}"
    evaluation.append_obs_to_file(pred_file, obs_label + "_stack", obs_pred)
    evaluation.append_obs_to_file(pred_file, obs_label + "_mean", np.mean(obs_pred, axis=0))


## DES Y3 

In [ ]:
wl_gamma_map, _ = catalog.build_metacal_map_from_cat(msfm_conf)
gc_count_map = catalog.build_maglim_map_from_cat(msfm_conf)

des_cl = preprocessing.get_preprocessed_cl_observation(
    wl_gamma_map=wl_gamma_map,
    gc_count_map=gc_count_map,
    # configuration
    msfm_conf=msfm_conf,
    dlss_conf=dlss_conf,
    base_dir=data_dir,
    nest_in=False,
    # selection
    with_lensing=with_lensing,
    with_clustering=with_clustering,
    with_cross_z=with_cross_z,
    with_cross_probe=with_cross_probe,
    # additional preprocessing
    apply_log=apply_log,
    standardize=standardize,
    make_plot=False,
)

des_pred = model(des_cl, training=False).numpy()
evaluation.plot_summary_space_prior_predictive(grid_preds, des_pred)

des_label = "obs/preds/DESy3"
evaluation.append_obs_to_file(pred_file, des_label, des_pred)

## Buzzard

In [ ]:
n_pix = msfm_conf["analysis"]["n_pix"]

def evaluate_buzzard(obs_label, lensing_file, clustering_file, with_lensing, with_clustering, nest_in=False, plot_diagnostics=False):
    # load the map
    if with_lensing:
        wl_map = buzzard.get_lensing_map(lensing_file, plot_diagnostics=plot_diagnostics)
    else:
        wl_map = np.zeros((n_pix,4,2))

    if with_clustering:
        gc_map = buzzard.get_clustering_map(clustering_file, plot_diagnostics=plot_diagnostics)
    else:
        gc_map = np.zeros((n_pix,4))

    # forward model
    obs_cl = preprocessing.get_preprocessed_cl_observation(
        wl_gamma_map=wl_map,
        gc_count_map=gc_map,
        # configuration
        msfm_conf=msfm_conf,
        dlss_conf=dlss_conf,
        base_dir=data_dir,
        nest_in=False,
        # selection
        with_lensing=with_lensing,
        with_clustering=with_clustering,
        with_cross_z=with_cross_z,
        with_cross_probe=(with_lensing and with_clustering),
        # additional preprocessing
        apply_log=apply_log,
        standardize=standardize,
        make_plot=False,
    )

    # evaluate
    obs_pred = model(obs_cl, training=False).numpy()
    print(obs_label, ": obs_pred =", obs_pred)

    if plot_diagnostics:
        plot_cls_space_prior_predictive(obs_cl)
        evaluation.plot_summary_space_prior_predictive(obs_pred)

    # save
    evaluation.append_obs_to_file(pred_file, obs_label, obs_pred)

    return obs_pred


In [ ]:
# buzzard_indices, lensing_files, clustering_files = buzzard.get_filenames()

# lensing_files = lensing_files[:2]

# buzzards = []
# for i, lensing_file, clustering_file in zip(buzzard_indices, lensing_files, clustering_files):
#     obs_label = f"obs/preds/Buzzard_{i}"
    
#     obs_pred = evaluate_buzzard(
#         obs_label, 
#         lensing_file, 
#         clustering_file, 
#         with_lensing=with_lensing,
#         with_clustering=with_clustering,
#         plot_diagnostics=True
#     )    
    
#     buzzards.append(obs_pred)

# buzzards = np.stack(buzzards, axis=0)
# evaluation.append_obs_to_file(pred_file, "Buzzard_stack", buzzards)
# evaluation.append_obs_to_file(pred_file, "Buzzard_mean", np.mean(buzzards, axis=0))
